# ⚖️ Contract Intelligence Engine — COMPLETE NOTEBOOK
### Hireathon 2026 — Problem Statement 3

---

## HOW TO RUN THIS NOTEBOOK

### First time (models not trained yet)
Run ALL cells top to bottom. Training takes ~40 min on T4 GPU.

### Returning session (models already trained)
Skip to **SECTION B** — just run Cells 10 onwards.

---

| Section | Cells | What it does |
|---------|-------|--------------|
| **A — Setup** | 1–3 | Install, restart, imports |
| **B — Data** | 4–5 | Build CSVs from CUAD |
| **C — Training** | 6–7 | Train classifier + QA model |
| **D — Verify** | 8 | Check all files saved |
| **E — Inference** | 9–16 | Load models, define all functions |
| **F — UI** | 17 | Launch Gradio UI |

---
## SECTION A — Setup

In [1]:
# CELL 1 — Install all packages
!pip uninstall -y transformers peft accelerate -q
!pip install -q \
  transformers==4.41.2 \
  peft==0.10.0 \
  accelerate==0.30.1 \
  datasets \
  pandas \
  scikit-learn \
  pymupdf \
  gradio
print('✅ All packages installed')
print('⚠️  NOW: Runtime → Restart Session — then run from Cell 3')

✅ All packages installed
⚠️  NOW: Runtime → Restart Session — then run from Cell 3


## ⚠️ CELL 2 — RESTART RUNTIME
Go to **Runtime → Restart Session**, then continue from Cell 3.
Do NOT run Cell 1 again after restarting.

In [2]:
# CELL 3 — All imports
import os, json, re, torch, fitz, hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer
)
import gradio as gr

print('✅ Imports OK')
print(f'   GPU: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

2026-03-28 13:49:02.863580: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774705743.027127     117 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774705743.074422     117 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774705743.442442     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774705743.442477     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774705743.442480     117 computation_placer.cc:177] computation placer alr

✅ Imports OK
   GPU: True | Device: Tesla T4


---
## SECTION B — Build Training Data from CUAD

In [10]:
# CELL 4 — Build multi-label CSV from CUAD
# Skips automatically if file already exists

CUAD_PATH = '/kaggle/input/datasets/ritviksingh12/contract/CUADv1.json'
CLF_CSV   = 'data/train_multi_clean.csv'

if os.path.exists(CLF_CSV):
    print(f'✅ {CLF_CSV} already exists — skipping')
    df_clf = pd.read_csv(CLF_CSV)
else:
    print('Building multi-label CSV...')
    with open(CUAD_PATH) as f:
        data = json.load(f)

    rows = []
    for doc in data['data']:
        for para in doc['paragraphs']:
            context = para['context']
            for qa in para['qas']:
                rows.append({
                    'text':     context[:512],
                    'question': qa['question'],
                    'label':    0 if qa['is_impossible'] else 1
                })

    df   = pd.DataFrame(rows)
    pivot = df.pivot_table(
        index='text', columns='question',
        values='label', fill_value=0
    ).reset_index()

    os.makedirs('data', exist_ok=True)
    pivot.to_csv('data/train_multi_raw.csv', index=False)

    df2 = pd.read_csv('data/train_multi_raw.csv')
    def clean_col(c):
        return c.split('"')[1] if '"' in c else c
    df2.columns = ['text'] + [clean_col(c) for c in df2.columns[1:]]
    df2.to_csv(CLF_CSV, index=False)
    df_clf = df2
    print(f'✅ Saved {CLF_CSV} — {len(df2)} rows, {len(df2.columns)-1} label cols')

Building multi-label CSV...
✅ Saved data/train_multi_clean.csv — 508 rows, 41 label cols


In [11]:
# CELL 5 — Build QA CSV from CUAD
# Skips automatically if file already exists

QA_CSV = 'data/train_qa.csv'

if os.path.exists(QA_CSV):
    print(f'✅ {QA_CSV} already exists — skipping')
else:
    print('Building QA CSV...')
    with open(CUAD_PATH) as f:
        data = json.load(f)

    rows = []
    for doc in data['data']:
        for para in doc['paragraphs']:
            context = para['context']
            for qa in para['qas']:
                if qa['is_impossible'] or not qa['answers']:
                    continue
                rows.append({
                    'context':      context,
                    'question':     qa['question'],
                    'answer_text':  qa['answers'][0]['text'],
                    'answer_start': qa['answers'][0]['answer_start']
                })

    pd.DataFrame(rows).to_csv(QA_CSV, index=False)
    print(f'✅ Saved {QA_CSV} — {len(rows)} QA pairs')

Building QA CSV...
✅ Saved data/train_qa.csv — 6702 QA pairs


---
## SECTION C — Train Models
**Skip this section if models already exist in `/kaggle/working/`**

In [12]:
# CELL 6 — Train multi-label classifier (Legal-BERT)
# Skips if already trained | Time: ~25 min on T4

CLF_SAVE = 'final_model_multi'

if os.path.exists(f'{CLF_SAVE}/model.safetensors'):
    print(f'✅ {CLF_SAVE}/ already exists — skipping training')
else:
    print('Training Legal-BERT classifier... (~25 min)')
    CLF_BASE = 'nlpaueb/legal-bert-base-uncased'
    clf_tok  = AutoTokenizer.from_pretrained(CLF_BASE)

    df_clf   = pd.read_csv(CLF_CSV)
    label_cols = df_clf.columns[1:].tolist()
    df_clf['labels'] = df_clf[label_cols].values.tolist()

    ds = Dataset.from_pandas(df_clf[['text', 'labels']])

    def tokenize(x):
        return clf_tok(x['text'], truncation=True, padding='max_length', max_length=256)

    ds = ds.map(tokenize, batched=True)
    ds = ds.train_test_split(test_size=0.1)

    clf_model = AutoModelForSequenceClassification.from_pretrained(
        CLF_BASE,
        num_labels=len(label_cols),
        problem_type='multi_label_classification'
    )

    Trainer(
        model=clf_model,
        args=TrainingArguments(
            output_dir='./results_clf',
            per_device_train_batch_size=16,
            num_train_epochs=3,
            fp16=True,
            save_strategy='no',
            report_to='none',
            logging_steps=100
        ),
        train_dataset=ds['train'],
        eval_dataset=ds['test']
    ).train()

    clf_model.save_pretrained(CLF_SAVE)
    clf_tok.save_pretrained(CLF_SAVE)
    print(f'✅ Classifier saved → {CLF_SAVE}/')

Training Legal-BERT classifier... (~25 min)


Map:   0%|          | 0/508 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss


✅ Classifier saved → final_model_multi/


In [14]:
# CELL 7 — Train QA model (BERT-SQuAD2)
# Skips if already trained | Time: ~15 min on T4

QA_SAVE = 'qa_model'

if os.path.exists(f'{QA_SAVE}/model.safetensors'):
    print(f'✅ {QA_SAVE}/ already exists — skipping training')
else:
    print('Training BERT-SQuAD2 QA model... (~15 min)')
    QA_BASE = 'deepset/bert-base-cased-squad2'
    qa_tok  = AutoTokenizer.from_pretrained(QA_BASE)

    df_qa = pd.read_csv(QA_CSV).dropna().head(20000)
    ds_qa = Dataset.from_pandas(df_qa)

    def preprocess_qa(example):
        inputs = qa_tok(
            example['question'], example['context'],
            truncation='only_second', padding='max_length',
            max_length=256, return_offsets_mapping=True
        )
        offsets    = inputs['offset_mapping']
        seq_ids    = inputs.sequence_ids()
        start_char = int(example['answer_start'])
        end_char   = start_char + len(str(example['answer_text']))
        s, e = 0, 0
        for i, (o, sid) in enumerate(zip(offsets, seq_ids)):
            if sid != 1:
                continue
            if o[0] <= start_char <= o[1]:
                s = i
            if o[0] <= end_char <= o[1]:
                e = i
        inputs['start_positions'] = s
        inputs['end_positions']   = e
        return inputs

    ds_qa = ds_qa.map(preprocess_qa, remove_columns=ds_qa.column_names)
    ds_qa = ds_qa.train_test_split(test_size=0.1)

    qa_model_train = AutoModelForQuestionAnswering.from_pretrained(QA_BASE)

    Trainer(
        model=qa_model_train,
        args=TrainingArguments(
            output_dir='./results_qa',
            per_device_train_batch_size=16,
            num_train_epochs=2,
            fp16=True,
            save_strategy='no',
            report_to='none',
            logging_steps=100
        ),
        train_dataset=ds_qa['train'],
        eval_dataset=ds_qa['test']
    ).train()

    qa_model_train.save_pretrained(QA_SAVE)
    qa_tok.save_pretrained(QA_SAVE)
    print(f'✅ QA model saved → {QA_SAVE}/')

✅ qa_model/ already exists — skipping training


---
## SECTION D — Verify All Files

In [15]:
# CELL 8 — Verify all required files exist before loading

required = {
    'Classifier model':   'final_model_multi/model.safetensors',
    'Classifier config':  'final_model_multi/config.json',
    'Classifier tokenizer': 'final_model_multi/tokenizer.json',
    'QA model':           'qa_model/model.safetensors',
    'QA config':          'qa_model/config.json',
    'QA tokenizer':       'qa_model/tokenizer.json',
    'Label CSV':          'data/train_multi_clean.csv',
}

all_ok = True
for label, path in required.items():
    full = f'/kaggle/working/{path}'
    exists = os.path.exists(full)
    size   = f'{os.path.getsize(full)/1e6:.1f} MB' if exists else 'MISSING'
    icon   = '✅' if exists else '❌'
    print(f'{icon}  {label:25s}  {size}')
    if not exists:
        all_ok = False

print()
if all_ok:
    print('🎉 All files ready — continue to Section E')
else:
    print('❌ Missing files — re-run Cells 6 and/or 7')

✅  Classifier model           438.1 MB
✅  Classifier config          0.0 MB
✅  Classifier tokenizer       0.7 MB
✅  QA model                   430.9 MB
✅  QA config                  0.0 MB
✅  QA tokenizer               0.7 MB
✅  Label CSV                  0.3 MB

🎉 All files ready — continue to Section E


---
## SECTION E — Load Models & Define All Functions
**Start here if models already trained from a previous run.**

In [16]:
# CELL 9 — Load both trained models for inference

CLF_PATH = '/kaggle/working/final_model_multi'
QA_PATH  = '/kaggle/working/qa_model'
DATA_CSV = '/kaggle/working/data/train_multi_clean.csv'

print('Loading classifier...')
clf_tokenizer = AutoTokenizer.from_pretrained(CLF_PATH)
clf_model     = AutoModelForSequenceClassification.from_pretrained(CLF_PATH)

print('Loading QA model...')
qa_tokenizer = AutoTokenizer.from_pretrained(QA_PATH)
qa_model     = AutoModelForQuestionAnswering.from_pretrained(QA_PATH)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
clf_model.to(device).eval()
qa_model.to(device).eval()

label_cols = pd.read_csv(DATA_CSV).columns[1:].tolist()
print(f'\n✅ Both models loaded on {device}')
print(f'   {len(label_cols)} clause label columns available')

Loading classifier...
Loading QA model...

✅ Both models loaded on cuda
   41 clause label columns available


In [17]:
# CELL 10 — 15 Clause Questions

clause_questions = {
    'Document Name':               'What is the title of this agreement?',
    'Parties':                     'This agreement is between which parties?',
    'Agreement Date':              'This agreement is dated when?',
    'Effective Date':              'When does this agreement become effective?',
    'Expiration Date':             'When does this agreement expire?',
    'Governing Law':               'This agreement is governed by which law?',
    'Cap On Liability':            'Is there a maximum cap on damages or liability?',
    'Uncapped Liability':          'Is liability stated to be unlimited?',
    'Termination for Convenience': 'Can either party terminate without cause?',
    'Change Of Control':           'What happens if ownership or control changes?',
    'Anti-Assignment':             'Can this agreement be assigned to another party?',
    'Audit Rights':                'Does either party have audit rights?',
    'Exclusivity':                 'Is this agreement exclusive?',
    'Non-Compete':                 'Is competition or solicitation restricted?',
    'Revenue/Profit Sharing':      'Is revenue or profit shared between parties?'
}
print(f'✅ {len(clause_questions)} clause types defined')

✅ 15 clause types defined


In [18]:
# CELL 11 — Rule-based extractors (instant, no model needed)

def extract_parties_rule(text):
    for pat in [
        r'(?:between|BETWEEN)\s+(.+?)\s+(?:and|AND)\s+(.+?)(?:\s*[\.,\(])',
        r'(?:by and between)\s+(.+?)\s+and\s+(.+?)(?:\s*[\.,\(])',
    ]:
        m = re.search(pat, text[:3000])
        if m:
            p1 = m.group(1).strip().strip('"()')
            p2 = m.group(2).strip().strip('"()')
            if 3 < len(p1) < 120 and 3 < len(p2) < 120:
                return f'{p1} and {p2}'
    return None

def extract_governing_law_rule(text):
    for pat in [
        r'governed by (?:the laws? of )?([A-Za-z][A-Za-z ,]+?)(?:\.|,|\s+and)',
        r'laws? of the [Ss]tate of ([A-Za-z ]+?)(?:\.|,)',
        r'jurisdiction of ([A-Za-z ]+?)(?:\.|,)',
    ]:
        m = re.search(pat, text)
        if m:
            r = m.group(1).strip()
            if 2 < len(r) < 80:
                return r
    return None

def extract_doc_name_rule(text):
    lines = [l.strip() for l in text.split('\n')[:20] if l.strip()]
    kw = ['agreement', 'contract', 'license', 'deed', 'addendum', 'amendment']
    for line in lines:
        if any(k in line.lower() for k in kw) and len(line) < 150:
            return line
    return lines[0] if lines else None

def extract_date_rule(text):
    for pat in [
        r'(?:dated?|as of|effective)\s+([A-Za-z]+ \d{1,2},?\s*\d{4})',
        r'(?:dated?|as of|effective)\s+(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})',
        r'(\d{1,2}(?:st|nd|rd|th)?\s+day of\s+[A-Za-z]+,?\s*\d{4})',
    ]:
        m = re.search(pat, text[:5000], re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return None

print('✅ Rule-based extractors ready')

✅ Rule-based extractors ready


In [19]:
# CELL 12 — QA extractor with confidence score + source snippet

def extract_qa_with_score(question, context):
    """
    Returns (answer, confidence_0_to_1, source_snippet) or (None, 0, None)
    """
    inputs = qa_tokenizer(
        question, context,
        return_tensors='pt',
        truncation=True,
        max_length=384
    ).to(device)

    with torch.no_grad():
        out = qa_model(**inputs)

    sl = out.start_logits[0]
    el = out.end_logits[0]

    best_score = -1e9
    best_s, best_e = 0, 0
    for s in range(len(sl)):
        for e in range(s, min(s + 40, len(el))):
            sc = sl[s].item() + el[e].item()
            if sc > best_score:
                best_score = sc
                best_s, best_e = s, e

    if best_score < 2.0:
        return None, 0.0, None

    answer = qa_tokenizer.decode(
        inputs['input_ids'][0][best_s:best_e + 1],
        skip_special_tokens=True
    ).strip()

    if len(answer) < 3 or len(answer) > 300:
        return None, 0.0, None
    if answer.lower() in {'the', 'a', 'an', 'and', 'or', 'of', 'is', 'are'}:
        return None, 0.0, None

    norm_score  = min(1.0, max(0.0, (best_score - 2.0) / 18.0))
    idx         = context.find(answer)
    snip_start  = max(0, idx - 60) if idx >= 0 else 0
    snippet     = '...' + context[snip_start:snip_start + 160].strip() + '...'

    return answer, round(norm_score, 2), snippet


def extract_qa(question, context):
    ans, _, _ = extract_qa_with_score(question, context)
    return ans

print('✅ QA extractor ready')

✅ QA extractor ready


In [20]:
# CELL 13 — Chunker (500 words, 20-word overlap)

def chunk_text(text, size=500, overlap=20):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(' '.join(words[i:i + size]))
        i += size - overlap
    return chunks

print('✅ Chunker ready')

✅ Chunker ready


In [21]:
# CELL 14 — Voting Analyzer + Risk Engine
#
# ACCURACY UPGRADE: collects answers from all chunks,
# picks most frequent answer per clause (majority vote)

def analyze_with_voting(text):
    results = {}

    # Pass 1 — rules (instant)
    for clause, val in [
        ('Document Name',  extract_doc_name_rule(text)),
        ('Parties',        extract_parties_rule(text)),
        ('Governing Law',  extract_governing_law_rule(text)),
        ('Agreement Date', extract_date_rule(text)),
    ]:
        if val:
            results[clause] = {
                'answer': val, 'confidence': 1.0,
                'source': 'Pattern matching', 'method': 'rule'
            }

    # Pass 2 — QA with voting
    remaining = [k for k in clause_questions if k not in results]
    if not remaining:
        return results

    chunks = chunk_text(text)
    chunks = chunks[:max(1, int(len(chunks) * 0.65))]

    votes = {c: [] for c in remaining}
    for chunk in chunks:
        still_needed = [c for c in remaining if c not in results]
        if not still_needed:
            break
        for clause in still_needed:
            ans, score, snip = extract_qa_with_score(clause_questions[clause], chunk)
            if ans:
                votes[clause].append((ans, score, snip))

    for clause in remaining:
        if not votes[clause]:
            continue
        best_answer = Counter(v[0] for v in votes[clause]).most_common(1)[0][0]
        best_entry  = max([v for v in votes[clause] if v[0] == best_answer], key=lambda x: x[1])
        results[clause] = {
            'answer':     best_answer,
            'confidence': best_entry[1],
            'source':     best_entry[2],
            'method':     'qa'
        }

    return results


# Risk engine — 10 checks
RISK_RULES = [
    (lambda r: 'Cap On Liability'            not in r, 'HIGH',   'No cap on liability — unlimited damages exposure'),
    (lambda r: 'Uncapped Liability'          in r,     'HIGH',   'Uncapped liability explicitly stated'),
    (lambda r: 'Termination for Convenience' not in r, 'MEDIUM', 'No termination for convenience clause'),
    (lambda r: 'Governing Law'               not in r, 'MEDIUM', 'Governing law not specified'),
    (lambda r: 'Expiration Date'             not in r, 'MEDIUM', 'No expiration date — contract may be open-ended'),
    (lambda r: 'Anti-Assignment'             not in r, 'LOW',    'No anti-assignment clause'),
    (lambda r: 'Audit Rights'                not in r, 'LOW',    'No audit rights clause'),
    (lambda r: 'Non-Compete'                 in r,     'NOTE',   'Non-compete clause present — review scope'),
    (lambda r: 'Exclusivity'                 in r,     'NOTE',   'Exclusivity restriction present'),
    (lambda r: 'Change Of Control'           not in r, 'LOW',    'No change of control clause'),
]
SEVERITY_EMOJI = {'HIGH': '🔴', 'MEDIUM': '🟡', 'LOW': '🔵', 'NOTE': '⚪'}

def risk_analysis(clause_results):
    keys  = set(clause_results.keys())
    risks = []
    for cond, sev, msg in RISK_RULES:
        if cond(keys):
            risks.append({'severity': sev, 'message': msg, 'emoji': SEVERITY_EMOJI[sev]})
    return risks

def risk_score(risks):
    w = {'HIGH': 30, 'MEDIUM': 15, 'LOW': 8, 'NOTE': 2}
    return min(100, sum(w.get(r['severity'], 0) for r in risks))

def full_analysis(text):
    clauses = analyze_with_voting(text)
    risks   = risk_analysis(clauses)
    return {'clauses': clauses, 'risks': risks, 'risk_score': risk_score(risks)}

print('✅ Voting analyzer + risk engine ready')

✅ Voting analyzer + risk engine ready


In [22]:
# CELL 15 — PDF extractor + MD5 session cache

_cache = {}

def pdf_to_text(path):
    doc = fitz.open(path)
    return '\n'.join(page.get_text() for page in doc)

def cached_analysis(path):
    key = hashlib.md5(open(path, 'rb').read()).hexdigest()
    if key not in _cache:
        _cache[key] = full_analysis(pdf_to_text(path))
    return _cache[key]

print('✅ PDF extractor + cache ready')

✅ PDF extractor + cache ready


In [23]:
# CELL 16 — Parallel bulk processor + UI HTML builders

def process_single(file):
    fname = os.path.basename(file.name)
    try:
        return fname, cached_analysis(file.name), None
    except Exception as e:
        return fname, None, str(e)

def process_parallel(files, max_workers=4):
    results = [None] * len(files)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        fut_idx = {ex.submit(process_single, f): i for i, f in enumerate(files)}
        for fut in as_completed(fut_idx):
            results[fut_idx[fut]] = fut.result()
    return results


def build_risk_html(filename, risks, score):
    color = '#e53e3e' if score >= 60 else '#d69e2e' if score >= 30 else '#38a169'
    rows  = ''.join(
        f'<tr><td style="padding:5px 10px">{r["emoji"]} <b>{r["severity"]}</b></td>'
        f'<td style="padding:5px 10px">{r["message"]}</td></tr>'
        for r in risks
    ) if risks else '<tr><td colspan="2" style="padding:5px 10px">✅ No major risks</td></tr>'

    return f"""
<div style="font-family:sans-serif;margin-bottom:20px;border:1px solid #e2e8f0;border-radius:8px;overflow:hidden">
  <div style="background:#2d3748;color:white;padding:10px 16px;font-weight:600;font-size:1em">{filename}</div>
  <div style="padding:12px 16px;background:#f7fafc">
    <div style="display:flex;align-items:center;gap:12px;margin-bottom:6px">
      <span style="font-weight:600">Risk Score:</span>
      <span style="font-size:1.3em;font-weight:700;color:{color}">{score}/100</span>
    </div>
    <div style="background:#e2e8f0;border-radius:4px;height:8px">
      <div style="background:{color};height:8px;width:{score}%;border-radius:4px"></div>
    </div>
  </div>
  <table style="width:100%;border-collapse:collapse;font-size:0.88em">{rows}</table>
</div>"""


def build_clause_html(clauses):
    cards = []
    for clause, data in clauses.items():
        if isinstance(data, dict):
            answer = data.get('answer', 'Not found')
            conf   = data.get('confidence', 0)
            source = data.get('source', '')
            method = data.get('method', 'qa')
        else:
            answer, conf, source, method = str(data), 1.0, '', 'rule'

        cp  = int(conf * 100)
        bc  = '#38a169' if conf > 0.6 else '#d69e2e' if conf > 0.3 else '#e53e3e'
        bdg = ('<span style="background:#e9d8fd;color:#553c9a;padding:1px 7px;'
               'border-radius:3px;font-size:0.75em">rule</span>'
               if method == 'rule' else
               '<span style="background:#bee3f8;color:#2a4365;padding:1px 7px;'
               'border-radius:3px;font-size:0.75em">QA model</span>')

        cards.append(f"""
<div style="border:1px solid #e2e8f0;border-radius:6px;padding:10px 14px;
             margin-bottom:8px;font-family:sans-serif">
  <div style="display:flex;justify-content:space-between;align-items:center">
    <b style="font-size:0.88em">{clause}</b> {bdg}
  </div>
  <div style="margin:4px 0;color:#2d3748;font-size:0.95em">{answer}</div>
  <div style="display:flex;align-items:center;gap:8px;margin-top:4px">
    <div style="background:#e2e8f0;border-radius:3px;height:5px;flex:1">
      <div style="background:{bc};height:5px;width:{cp}%;border-radius:3px"></div>
    </div>
    <span style="font-size:0.75em;color:#718096">{cp}%</span>
  </div>
  <div style="font-size:0.72em;color:#a0aec0;margin-top:3px">{(source or '')[:120]}</div>
</div>""")
    return ''.join(cards)


print('✅ Parallel processor + UI helpers ready')

✅ Parallel processor + UI helpers ready


---
## SECTION F — Launch Gradio UI

In [24]:
# CELL 17 — Launch enhanced Gradio UI

# ── Gradio callbacks ─────────────────────────────────────

def run_analysis(files):
    """Tab 1 — Bulk analysis with parallel processing."""
    if not files:
        return None, '<p>No files uploaded.</p>', '<p>No files uploaded.</p>'

    rows, risk_html, detail_html = [], '', ''

    for fname, res, err in process_parallel(files):
        if err:
            risk_html += f'<p style="color:red">❌ {fname}: {err}</p>'
            continue

        clauses = res['clauses']
        risks   = res['risks']
        score   = res['risk_score']

        row = {'Contract': fname}
        for k, v in clauses.items():
            row[k] = v['answer'] if isinstance(v, dict) else v
        rows.append(row)

        risk_html   += build_risk_html(fname, risks, score)
        detail_html += f'<h3 style="font-family:sans-serif;margin-top:20px">{fname}</h3>'
        detail_html += build_clause_html(clauses)

    df = pd.DataFrame(rows).fillna('Not found')
    return df, risk_html, detail_html


def run_qa(question, files):
    """Tab 2 — NL question, returns best answer with confidence."""
    if not files or not question.strip():
        return '<p>Upload a PDF and type a question.</p>'

    text = pdf_to_text(files[0].name)
    best_ans, best_score, best_snip = None, 0, None

    for chunk in chunk_text(text):
        ans, score, snip = extract_qa_with_score(question, chunk)
        if ans and score > best_score:
            best_ans, best_score, best_snip = ans, score, snip

    if best_ans:
        cp = int(best_score * 100)
        bc = '#38a169' if best_score > 0.6 else '#d69e2e'
        return f"""
<div style="font-family:sans-serif;border:1px solid #e2e8f0;border-radius:8px;padding:16px">
  <div style="font-weight:600;margin-bottom:8px">Answer</div>
  <div style="font-size:1.15em;color:#2d3748;margin-bottom:12px">{best_ans}</div>
  <div style="display:flex;align-items:center;gap:8px;margin-bottom:10px">
    <span style="font-size:0.85em;color:#718096">Confidence</span>
    <div style="background:#e2e8f0;border-radius:3px;height:6px;width:200px">
      <div style="background:{bc};height:6px;width:{cp}%;border-radius:3px"></div>
    </div>
    <span style="font-size:0.85em">{cp}%</span>
  </div>
  <div style="font-size:0.8em;color:#718096;background:#f7fafc;padding:8px;border-radius:4px">
    {best_snip or ''}
  </div>
</div>"""

    return '<p style="color:#e53e3e">❌ No clear answer found in this contract.</p>'


def run_compare(files):
    """Tab 3 — Side-by-side clause comparison."""
    if not files or len(files) < 2:
        return '<p>Upload at least 2 contracts to compare.</p>'

    results = process_parallel(files)
    fnames  = [r[0] for r in results]

    header_cells = '<th style="padding:8px 12px;text-align:left">Clause</th>' + \
        ''.join(f'<th style="padding:8px 12px;max-width:200px">{n}</th>' for n in fnames)

    body = ''
    for clause in clause_questions:
        cells = f'<td style="font-weight:600;padding:6px 10px;font-size:0.85em">{clause}</td>'
        for _, res, err in results:
            if err or res is None:
                cells += '<td style="background:#fed7d7;padding:6px 10px;color:#c53030;font-size:0.8em">Error</td>'
                continue
            data = res['clauses'].get(clause)
            if data:
                ans  = data['answer'] if isinstance(data, dict) else data
                conf = data.get('confidence', 1.0) if isinstance(data, dict) else 1.0
                bg   = '#c6f6d5' if conf > 0.6 else '#fefcbf'
                cells += f'<td style="background:{bg};padding:6px 10px;font-size:0.8em">{ans[:80]}</td>'
            else:
                cells += '<td style="background:#fed7d7;padding:6px 10px;color:#c53030;font-size:0.8em">Not found</td>'
        body += f'<tr>{cells}</tr>'

    return f"""
<div style="overflow-x:auto">
<table style="border-collapse:collapse;font-family:sans-serif;width:100%">
  <thead style="background:#2d3748;color:white"><tr>{header_cells}</tr></thead>
  <tbody>{body}</tbody>
</table>
</div>
<div style="font-size:0.75em;color:#718096;margin-top:8px;font-family:sans-serif">
  🟢 High confidence &nbsp;|&nbsp; 🟡 Medium confidence &nbsp;|&nbsp; 🔴 Not found
</div>"""


# ── Build UI ─────────────────────────────────────────────

css = ".gradio-container{max-width:1100px!important} footer{display:none!important}"

with gr.Blocks(title='Contract Intelligence Engine', css=css) as demo:

    gr.HTML("""
    <div style="background:#1a202c;color:white;padding:20px 28px;
                border-radius:8px;margin-bottom:16px;font-family:sans-serif">
      <div style="font-size:1.6em;font-weight:700">⚖️ Contract Intelligence Engine</div>
      <div style="color:#a0aec0;margin-top:4px;font-size:0.95em">
        Parallel processing · Voting accuracy · 10-point risk scoring · Clause confidence bars
      </div>
    </div>
    """)

    # ── Tab 1: Bulk Analysis ──────────────────────────────
    with gr.Tab('📄 Bulk Analysis'):
        gr.Markdown(
            'Upload **one or more** PDF contracts. '
            'All contracts are analyzed in parallel. '
            'Results show extracted clauses with confidence scores.'
        )
        with gr.Row():
            t1_upload = gr.File(
                label='Upload contracts (PDF)',
                file_count='multiple',
                file_types=['.pdf'],
                scale=3
            )
            with gr.Column(scale=1):
                t1_btn = gr.Button('🔍 Analyze All', variant='primary', size='lg')
                gr.Markdown(
                    '_Sample contracts are in:_\n'
                    '`/kaggle/input/datasets/.../contracts/`'
                )

        with gr.Tabs():
            with gr.Tab('📊 Clause Table'):
                t1_table = gr.Dataframe(label='Extracted clauses — one row per contract', wrap=True)
            with gr.Tab('🔴 Risk Report'):
                t1_risk = gr.HTML()
            with gr.Tab('📋 Clause Detail + Confidence'):
                t1_detail = gr.HTML()

        t1_btn.click(fn=run_analysis, inputs=[t1_upload], outputs=[t1_table, t1_risk, t1_detail])

    # ── Tab 2: Ask a Question ─────────────────────────────
    with gr.Tab('💬 Ask a Question'):
        gr.Markdown(
            'Ask any natural language question about a contract. '
            'The system searches all chunks and returns the highest-confidence answer.\n\n'
            '**Example questions:**\n'
            '- *Does this contract have a non-compete clause?*\n'
            '- *What is the governing law?*\n'
            '- *Can either party terminate without cause?*\n'
            '- *What happens on change of control?*'
        )
        with gr.Row():
            t2_upload = gr.File(
                label='Upload contract (PDF)',
                file_count='multiple',
                file_types=['.pdf'],
                scale=1
            )
            with gr.Column(scale=2):
                t2_question = gr.Textbox(
                    label='Your question',
                    placeholder='Does this contract have a non-compete clause?',
                    lines=3
                )
                t2_btn = gr.Button('💬 Ask', variant='primary')
        t2_out = gr.HTML()
        t2_btn.click(fn=run_qa, inputs=[t2_question, t2_upload], outputs=[t2_out])

    # ── Tab 3: Compare ────────────────────────────────────
    with gr.Tab('🔀 Compare Contracts'):
        gr.Markdown(
            'Upload **2–5 contracts** to see a side-by-side comparison of all 15 clause types.\n'
            'Green = found with high confidence · Yellow = found with medium confidence · Red = missing'
        )
        t3_upload = gr.File(
            label='Upload 2–5 contracts (PDF)',
            file_count='multiple',
            file_types=['.pdf']
        )
        t3_btn = gr.Button('🔀 Compare', variant='primary')
        t3_out = gr.HTML()
        t3_btn.click(fn=run_compare, inputs=[t3_upload], outputs=[t3_out])

demo.launch(share=True)
# share=True gives a public gradio.live URL that works inside Kaggle

/tmp/ipykernel_117/4001849275.py:113: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title='Contract Intelligence Engine', css=css) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://38bcab93c8c651617e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
